# RL Algorithm Comparison

This notebook compares fundamental RL algorithms on the **same GridWorld** problem:

| Algorithm | Type | Needs model? | How it learns |
|-----------|------|-------------|---------------|
| **Value Iteration** | DP | Yes (T, R) | Iterates Bellman optimality equation over all states |
| **Policy Iteration** | DP | Yes (T, R) | Alternates policy evaluation and improvement |
| **Q-Learning** | Model-free | No | Off-policy TD — updates toward `max Q(s')` |
| **SARSA** | Model-free | No | On-policy TD — updates toward `Q(s', a')` actually taken |
| **Dyna-Q** | Model-based | Learns one | Q-learning + simulated replay from a learned model |
| **DQN** | Model-free + NN | No | Neural network replaces Q-table; experience replay + target network |
| **Double DQN** | Model-free + NN | No | DQN + decoupled action selection/evaluation to reduce overestimation |

**Goal:** See how each algorithm finds the optimal policy, how fast it converges,
and what trade-offs each makes.  VI/PI are the "ground truth" — they compute the
exact optimal policy because they have full access to the MDP.  The learning agents
must discover the same policy through trial and error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Our GridWorld and agent implementations live in the same directory
from gridworld import GridWorld
from agents.dp import ValueIteration, PolicyIteration
from agents.tabular import TabularAgent
from agents.dqn import DQNAgent

print("All imports OK.")

## Environment Setup

We use a grid with traps to make the problem interesting — the agent
must learn to navigate around obstacles to reach the goal.
Feel free to tweak the grid dimensions, trap locations, and reward
values to see how each algorithm adapts.

In [ ]:
env = GridWorld(
    rows=6, cols=6,
    goal=(0, 5),
    traps={(1, 4), (1, 5)},
    start=(5, 0),
    goal_reward=10.0,
    trap_reward=-10.0,
    step_cost=-0.1,
)

print(env)
env.show_grid()

---

## Section 1: Dynamic Programming — Value Iteration & Policy Iteration

These are the **ground truth** solvers.  They require full knowledge of the MDP
(transition function T and reward function R) and compute the exact optimal
policy by iterating the Bellman equations.

- **Value Iteration** repeatedly applies `V(s) = max_a [R(s,a) + γ V(s')]`
  until V converges, then extracts the greedy policy.
- **Policy Iteration** alternates between evaluating the current policy
  (computing V^π) and improving it (making π greedy w.r.t. V).  Typically
  fewer outer iterations but each is more expensive.

In [ ]:
# --- Value Iteration ---
vi = ValueIteration(env, gamma=0.95)
vi_iters = vi.solve()
print(f"Value Iteration converged in {vi_iters} iterations.")
print()
vi.show()

In [ ]:
# --- Policy Iteration ---
pi = PolicyIteration(env, gamma=0.95)
pi_iters = pi.solve()
print(f"Policy Iteration converged in {pi_iters} improvement rounds.")
print()
pi.show()

In [ ]:
# --- Convergence: Bellman residual per VI iteration ---
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(vi.history) + 1), vi.history, 'o-', color='tab:blue')
plt.axhline(y=1e-6, color='gray', linestyle='--', label='theta (convergence threshold)')
plt.xlabel('Iteration')
plt.ylabel('Max Bellman Residual (delta)')
plt.title('Value Iteration Convergence')
plt.yscale('log')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Do VI and PI agree? ---
policies_match = np.array_equal(vi.policy, pi.policy)
values_close = np.allclose(vi.V, pi.V, atol=1e-4)

print(f"Policies identical:  {policies_match}")
print(f"Values close:        {values_close}")
print()
if policies_match:
    print("Both methods found the same optimal policy — as expected.")
    print("VI needed", vi_iters, "sweeps; PI needed", pi_iters, "policy rounds")
    print(f"(PI evaluation iterations per round: {pi.eval_iterations})")
else:
    print("WARNING: policies differ — check for bugs!")

---

## Section 2: Model-Free Methods — Q-Learning & SARSA

Now we **don't** give the agent the model.  It must learn Q(s, a) from
actual experience — taking actions, observing rewards, and updating.

- **Q-Learning** (off-policy): always updates toward `max_a' Q(s', a')`,
  so it converges to Q* regardless of how the agent explores.
- **SARSA** (on-policy): updates toward `Q(s', a')` where a' is the action
  actually taken next.  This means SARSA's learned policy accounts for
  exploration noise — it finds a safer path that avoids states where a
  random slip would be catastrophic (like trap cells).

In [ ]:
N_EPISODES = 200
GAMMA = 0.95
ALPHA = 0.1
EPSILON = 0.1

# --- Q-Learning ---
ql = TabularAgent(env, epsilon=EPSILON, gamma=GAMMA, alpha=ALPHA, seed=42)
ql_steps = ql.run_episodes(N_EPISODES)
print(f"Q-Learning: {N_EPISODES} episodes, {ql.total_steps} total steps")
print(f"  Last 10 episodes: {ql_steps[-10:]}")
print()
ql.show()

In [ ]:
# --- SARSA (on_policy=True is the ONLY difference from Q-learning) ---
sarsa = TabularAgent(env, on_policy=True, epsilon=EPSILON, gamma=GAMMA, alpha=ALPHA, seed=42)
sarsa_steps = sarsa.run_episodes(N_EPISODES)
print(f"SARSA: {N_EPISODES} episodes, {sarsa.total_steps} total steps")
print(f"  Last 10 episodes: {sarsa_steps[-10:]}")
print()
sarsa.show()

In [ ]:
# --- Learning curves: steps per episode ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw steps
axes[0].plot(ql_steps, alpha=0.3, color='tab:blue', label='Q-Learning')
axes[0].plot(sarsa_steps, alpha=0.3, color='tab:orange', label='SARSA')

# Smoothed (rolling average over 10 episodes)
window = 10
ql_smooth = np.convolve(ql_steps, np.ones(window)/window, mode='valid')
sarsa_smooth = np.convolve(sarsa_steps, np.ones(window)/window, mode='valid')
axes[0].plot(range(window-1, N_EPISODES), ql_smooth, color='tab:blue', linewidth=2)
axes[0].plot(range(window-1, N_EPISODES), sarsa_smooth, color='tab:orange', linewidth=2)

axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Steps to goal')
axes[0].set_title('Learning Curves: Steps per Episode')
axes[0].legend()

# Cumulative V*(start) over episodes
# We track V*(start) = max_a Q(start, a) after each episode as a proxy
# for how much value has propagated back to the starting state.
# Re-run with tracking to get this.

def run_with_v_tracking(env, n_episodes, **kwargs):
    """Run a TabularAgent and record V*(start) after each episode."""
    agent = TabularAgent(env, **kwargs)
    v_start = []
    steps_list = []
    for _ in range(n_episodes):
        steps = agent.run_episode()
        steps_list.append(steps)
        v_start.append(np.max(agent.Q[env.start]))
    return agent, steps_list, v_start

_, ql_steps2, ql_vstart = run_with_v_tracking(
    env, N_EPISODES,
    epsilon=EPSILON, gamma=GAMMA, alpha=ALPHA, seed=42)
_, sarsa_steps2, sarsa_vstart = run_with_v_tracking(
    env, N_EPISODES, on_policy=True,
    epsilon=EPSILON, gamma=GAMMA, alpha=ALPHA, seed=42)

# V*(start) from VI as reference line
vi_v_start = vi.V[env.start]

axes[1].plot(ql_vstart, color='tab:blue', label='Q-Learning')
axes[1].plot(sarsa_vstart, color='tab:orange', label='SARSA')
axes[1].axhline(y=vi_v_start, color='gray', linestyle='--',
                label=f'VI optimal ({vi_v_start:.2f})')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('V*(start) = max_a Q(start, a)')
axes[1].set_title('Value Propagation to Start State')
axes[1].legend()

plt.tight_layout()
plt.show()

### Q-Learning vs. SARSA: What to look for

- **Q-Learning** converges to the **optimal** policy (same as VI/PI) even
  though it explores with ε-greedy.  Its V*(start) should approach the VI line.
- **SARSA** converges to a policy that's optimal *under the current exploration*.
  Near traps, SARSA learns to give those states a wider berth because it
  "knows" it will sometimes slip into them randomly (ε exploration).  This
  makes SARSA safer but slightly suboptimal in terms of shortest path.
- The step-to-goal curves show sample efficiency — how quickly each agent
  learns to reach the goal without wandering.

---

## Section 3: Head-to-Head Comparison

Now we put all algorithms side by side: the DP solvers (which compute
the answer immediately) and the learning agents (which improve over episodes).
We also include Dyna-Q with prioritized sweeping to show the effect of
model-based replay on learning speed.

In [ ]:
N_EPISODES_COMPARE = 200

# All six configs are the SAME class — only the flags differ.
configs = {
    'Q-Learning':   dict(epsilon=EPSILON, gamma=GAMMA, alpha=ALPHA, seed=42),
    'SARSA':        dict(on_policy=True, epsilon=EPSILON, gamma=GAMMA, alpha=ALPHA, seed=42),
    'Dyna k=5':     dict(k_sim=5,  epsilon=EPSILON, gamma=GAMMA, alpha=ALPHA, seed=42),
    'Dyna k=50':    dict(k_sim=50, epsilon=EPSILON, gamma=GAMMA, alpha=ALPHA, seed=42),
    'PriSweep k=5': dict(k_sim=5,  prioritized=True, epsilon=EPSILON, gamma=GAMMA, alpha=ALPHA, seed=42),
    'PriSweep k=50':dict(k_sim=50, prioritized=True, epsilon=EPSILON, gamma=GAMMA, alpha=ALPHA, seed=42),
}

results = {}   # name -> (agent, step_list, v_start_list)
for name, kwargs in configs.items():
    agent, steps, vstart = run_with_v_tracking(
        env, N_EPISODES_COMPARE, **kwargs)
    results[name] = (agent, steps, vstart)
    print(f"{name:20s}  total_steps={agent.total_steps:6d}  "
          f"last_10_avg={np.mean(steps[-10:]):5.1f}")

# Also get the DP optimal step count for reference
vi_optimal_steps = vi.evaluate_policy(n_episodes=1, max_steps=200)[0]
print(f"\nVI optimal path length: {vi_optimal_steps} steps")

In [ ]:
# --- Big comparison plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {
    'Q-Learning':    'tab:blue',
    'SARSA':         'tab:orange',
    'Dyna k=5':      'tab:green',
    'Dyna k=50':     'tab:green',
    'PriSweep k=5':  'tab:red',
    'PriSweep k=50': 'tab:red',
}
styles = {
    'Q-Learning':    '-',
    'SARSA':         '-',
    'Dyna k=5':      '--',
    'Dyna k=50':     '-',
    'PriSweep k=5':  '--',
    'PriSweep k=50': '-',
}

window = 10
for name, (agent, steps, vstart) in results.items():
    c, ls = colors[name], styles[name]
    smoothed = np.convolve(steps, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, N_EPISODES_COMPARE), smoothed,
                 color=c, linestyle=ls, linewidth=2, label=name)

axes[0].axhline(y=vi_optimal_steps, color='gray', linestyle=':',
                linewidth=2, label=f'VI optimal ({vi_optimal_steps} steps)')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Steps to goal (smoothed)')
axes[0].set_title('Learning Speed: Steps per Episode')
axes[0].legend(fontsize=8)
axes[0].set_ylim(bottom=0)

# V*(start) comparison
for name, (agent, steps, vstart) in results.items():
    c, ls = colors[name], styles[name]
    axes[1].plot(vstart, color=c, linestyle=ls, linewidth=2, label=name)

axes[1].axhline(y=vi_v_start, color='gray', linestyle=':',
                linewidth=2, label=f'VI optimal ({vi_v_start:.2f})')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('V*(start) = max_a Q(start, a)')
axes[1].set_title('Value Propagation to Start State')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# --- Side-by-side policy grids ---
print("=" * 80)
print("OPTIMAL POLICY (Value Iteration)")
print("=" * 80)
vi.show()

for name, (agent, steps, vstart) in results.items():
    print("=" * 80)
    print(f"{name.upper()}  (after {N_EPISODES_COMPARE} episodes)")
    print("=" * 80)
    agent.show()

In [ ]:
# --- Summary table ---
print(f"{'Algorithm':20s} {'Total Steps':>12s} {'Last-10 Avg':>12s} "
      f"{'V*(start)':>10s} {'Needs Model?':>13s}")
print("-" * 70)
print(f"{'VI (exact)':20s} {'N/A':>12s} {vi_optimal_steps:12d} "
      f"{vi_v_start:10.2f} {'Yes (given)':>13s}")
print(f"{'PI (exact)':20s} {'N/A':>12s} {vi_optimal_steps:12d} "
      f"{pi.V[env.start]:10.2f} {'Yes (given)':>13s}")

for name, (agent, steps, vstart) in results.items():
    needs_model = 'Learns one' if agent.k_sim > 0 else 'No'
    print(f"{name:20s} {agent.total_steps:12d} {np.mean(steps[-10:]):12.1f} "
          f"{vstart[-1]:10.2f} {needs_model:>13s}")

### Key takeaways

1. **VI and PI agree** — both compute the exact optimal policy.  PI uses fewer
   outer iterations but more work per iteration (policy evaluation to convergence).

2. **Q-Learning converges to the optimal policy** (same as VI), but needs many
   real interactions to propagate value from the goal backward through the grid.

3. **SARSA is more conservative** — near traps, it learns a safer route because
   it accounts for its own exploration noise.  Lower V*(start) reflects this trade-off.

4. **Dyna dramatically speeds up learning** by replaying stored transitions.
   More simulated updates (k=50 vs k=5) means faster convergence — each real
   step generates 50 "free" learning updates.

5. **Prioritized Sweeping is even faster** because it updates the most "wrong"
   estimates first, propagating value backward efficiently instead of randomly.

6. **The trade-off is computation vs. experience**: DP is cheapest in experience
   (zero!) but requires the model.  Pure Q-learning needs the most experience.
   Dyna sits in between — it learns a model from experience, then replays from it.

---

## Section 4: Function Approximation — DQN & Double DQN

Everything above used a **Q-table**: one number stored per (state, action) pair.
That works when you can enumerate all states (our 6x6 grid has 36), but breaks
completely when the state space is large or continuous (CartPole, Atari, robotics).

**DQN** replaces the Q-table with a neural network Q_θ(s) that takes a state and
outputs Q-values for all actions.  Three ingredients stabilize training:

1. **Experience replay** — store transitions, sample random batches (breaks correlation)
2. **Target network** — a frozen copy of Q_θ, updated periodically (stabilizes TD target)
3. **ε decay** — start with high exploration, gradually become greedy

**Double DQN** adds one small fix: instead of using the target network to both
*select* and *evaluate* the best next action (which overestimates), the online
network selects and the target network evaluates.  ~3 lines of code, measurable improvement.

On our GridWorld, DQN is overkill — but that's the point.  We can verify it
learns the same optimal policy as VI, and see exactly where function approximation
helps (generalization) and where it hurts (slower convergence on tiny problems).

In [ ]:
N_EPISODES_DQN = 300  # DQN needs more episodes than tabular (NN takes longer to fit)

# --- Vanilla DQN ---
dqn = DQNAgent(env, double=False, hidden=64, lr=1e-3, gamma=GAMMA,
               epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=2000,
               batch_size=32, target_update=100, seed=42)
dqn_steps = dqn.run_episodes(N_EPISODES_DQN)
print(f"DQN: {N_EPISODES_DQN} episodes, {dqn.total_steps} total steps")
print(f"  Last 10 episodes: {dqn_steps[-10:]}")
print()
dqn.show()

In [ ]:
# --- Double DQN ---
ddqn = DQNAgent(env, double=True, hidden=64, lr=1e-3, gamma=GAMMA,
                epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=2000,
                batch_size=32, target_update=100, seed=42)
ddqn_steps = ddqn.run_episodes(N_EPISODES_DQN)
print(f"Double DQN: {N_EPISODES_DQN} episodes, {ddqn.total_steps} total steps")
print(f"  Last 10 episodes: {ddqn_steps[-10:]}")
print()
ddqn.show()

In [ ]:
# --- DQN learning curves vs. tabular Q-learning ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Re-run tabular Q-learning for the same number of episodes for fair comparison
ql_long = TabularAgent(env, epsilon=EPSILON, gamma=GAMMA, alpha=ALPHA, seed=42)
ql_long_steps = ql_long.run_episodes(N_EPISODES_DQN)

window = 10

# Panel 1: Steps per episode
for label, steps, color, ls in [
    ('Q-Learning (tabular)', ql_long_steps, 'tab:blue', '-'),
    ('DQN',                  dqn_steps,     'tab:purple', '-'),
    ('Double DQN',           ddqn_steps,    'tab:cyan', '--'),
]:
    smoothed = np.convolve(steps, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(steps)), smoothed,
                 color=color, linestyle=ls, linewidth=2, label=label)
axes[0].axhline(y=vi_optimal_steps, color='gray', linestyle=':',
                linewidth=2, label=f'VI optimal ({vi_optimal_steps} steps)')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Steps to goal (smoothed)')
axes[0].set_title('Learning Speed: Tabular vs. DQN')
axes[0].legend(fontsize=8)
axes[0].set_ylim(bottom=0)

# Panel 2: V*(start) over episodes — track from fresh runs
def run_dqn_with_tracking(env, n_episodes, **kwargs):
    agent = DQNAgent(env, **kwargs)
    v_start, steps_list = [], []
    for _ in range(n_episodes):
        steps = agent.run_episode()
        steps_list.append(steps)
        v_start.append(np.max(agent.Q[env.start]))
    return agent, steps_list, v_start

_, _, dqn_vstart = run_dqn_with_tracking(
    env, N_EPISODES_DQN, double=False, seed=42, gamma=GAMMA)
_, _, ddqn_vstart = run_dqn_with_tracking(
    env, N_EPISODES_DQN, double=True, seed=42, gamma=GAMMA)
# Tabular Q-learning V*(start)
ql_track = TabularAgent(env, epsilon=EPSILON, gamma=GAMMA, alpha=ALPHA, seed=42)
ql_vstart_long = []
for _ in range(N_EPISODES_DQN):
    ql_track.run_episode()
    ql_vstart_long.append(np.max(ql_track.Q[env.start]))

for label, vstart, color, ls in [
    ('Q-Learning (tabular)', ql_vstart_long, 'tab:blue', '-'),
    ('DQN',                  dqn_vstart,     'tab:purple', '-'),
    ('Double DQN',           ddqn_vstart,    'tab:cyan', '--'),
]:
    axes[1].plot(vstart, color=color, linestyle=ls, linewidth=2, label=label)
axes[1].axhline(y=vi_v_start, color='gray', linestyle=':',
                linewidth=2, label=f'VI optimal ({vi_v_start:.2f})')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('V*(start)')
axes[1].set_title('Value Propagation: Tabular vs. DQN')
axes[1].legend(fontsize=8)

# Panel 3: DQN training loss
loss_window = 50
for label, agent, color, ls in [
    ('DQN',        dqn,  'tab:purple', '-'),
    ('Double DQN', ddqn, 'tab:cyan', '--'),
]:
    if agent.train_losses:
        smoothed = np.convolve(agent.train_losses,
                               np.ones(loss_window)/loss_window, mode='valid')
        axes[2].plot(smoothed, color=color, linestyle=ls, linewidth=1.5, label=label)
axes[2].set_xlabel('Training step')
axes[2].set_ylabel('MSE Loss (smoothed)')
axes[2].set_title('DQN Training Loss')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# --- Side-by-side policy comparison: VI vs. Tabular Q-learning vs. DQN vs. Double DQN ---
print("=" * 80)
print("GROUND TRUTH (Value Iteration)")
print("=" * 80)
vi.show()

print("=" * 80)
print("TABULAR Q-LEARNING (after 300 episodes)")
print("=" * 80)
ql_long.show()

print("=" * 80)
print("DQN (after 300 episodes)")
print("=" * 80)
dqn.show()

print("=" * 80)
print("DOUBLE DQN (after 300 episodes)")
print("=" * 80)
ddqn.show()

### DQN takeaways

1. **DQN learns the same optimal policy** as tabular Q-learning and VI — the
   neural network successfully approximates the Q-table.  On this small grid,
   it converges somewhat slower because the network must generalize (and sometimes
   over-generalizes) across states that a table would treat independently.

2. **Double DQN reduces overestimation**.  Compare the V*(s) grids: vanilla DQN
   may show inflated values in some cells (the network confidently predicts high
   Q-values for actions it hasn't explored well).  Double DQN's decoupled
   selection/evaluation keeps values more honest.

3. **Training loss drops but stays noisy** — this is normal for RL.  Unlike
   supervised learning, the target itself keeps changing (bootstrapping), and
   the data distribution shifts as the policy improves.  The target network
   helps, but doesn't eliminate this non-stationarity.

4. **Why DQN matters**: on this GridWorld it's overkill (36 states fit in a table).
   But CartPole has a continuous 4D state space — no table can enumerate all states.
   DQN's generalization is essential there: seeing one state teaches the network
   about similar states it hasn't visited yet.

---

## Section 5: Stochastic Scenarios

Everything above used a small, deterministic grid.  Now we crank up the complexity
to expose real algorithmic differences:

- **Scenario A — "The Gauntlet"** (Risk vs. Speed): two routes to the goal — a fast
  one through stochastic traps, and a safe one around the outside.  Does the agent
  learn to take the shortcut or play it safe?
- **Scenario B — "The Minefield"** (Scattered Risk): stochastic traps with varying
  probabilities scattered across the grid, plus 20% action slip.  Tests risk-weighted
  planning and generalization.
- **Scenario C — "The Maze"** (Planning Depth): a winding maze with walls and no
  stochasticity.  Pure test of how fast value propagates through long corridors.

All environments are 10x10 and use the same `GridWorld` class — just with walls,
`slip_prob`, and `stochastic_traps` enabled.

In [ ]:
# === Scenario definitions ===

gauntlet = GridWorld(
    rows=10, cols=10, goal=(0, 9), start=(9, 0),
    walls={
        (2,0),(2,1),(2,2),(2,3),          (2,6),(2,7),(2,8),(2,9),
        (6,0),(6,1),(6,2),(6,3),          (6,6),(6,7),(6,8),(6,9),
    },
    stochastic_traps={
        (4,1): (-10, 0.4), (4,2): (-10, 0.4), (4,3): (-10, 0.4),
        (4,4): (-10, 0.4), (4,5): (-10, 0.4), (4,6): (-10, 0.4),
        (4,7): (-10, 0.4),
    },
    slip_prob=0.1,
)

minefield = GridWorld(
    rows=10, cols=10, goal=(0, 9), start=(9, 0),
    walls={(2,4), (4,2), (4,7), (6,4), (6,8)},
    stochastic_traps={
        (1,2): (-10, 0.2), (1,6): (-10, 0.5),
        (2,1): (-10, 0.3), (2,8): (-10, 0.4),
        (3,3): (-10, 0.6), (3,6): (-10, 0.2),
        (5,1): (-10, 0.3), (5,5): (-10, 0.5),
        (6,3): (-10, 0.4), (6,7): (-10, 0.6),
        (7,5): (-10, 0.2),
        (8,1): (-10, 0.3), (8,7): (-10, 0.5),
    },
    slip_prob=0.2,
)

maze = GridWorld(
    rows=10, cols=10, goal=(0, 9), start=(9, 0),
    walls={
        (0,2), (1,2), (1,4),(1,5),(1,6),(1,7),
        (2,2), (2,7),
        (3,2),(3,3),(3,4),(3,5), (3,7),
        (4,5),
        (5,1),(5,2),(5,3), (5,5),(5,6),(5,7),(5,8),
        (6,1), (6,8),
        (7,1), (7,3),(7,4),(7,5),(7,6),(7,7),
        (8,3), (8,8),
    },
    slip_prob=0.0,
)

scenarios = [
    ("A: The Gauntlet (risk vs. speed)", gauntlet),
    ("B: The Minefield (scattered risk)", minefield),
    ("C: The Maze (planning depth)",      maze),
]

for name, sc in scenarios:
    print(f"=== Scenario {name} ===")
    print(sc)
    sc.show_grid()

In [ ]:
# === Solve each scenario with VI (ground truth) ===

vi_results = {}
for name, sc in scenarios:
    vi_sc = ValueIteration(sc, gamma=0.95)
    vi_sc.solve()
    vi_results[name] = vi_sc
    print(f"--- {name} ---")
    print(f"VI converged in {vi_sc.iterations} iterations")
    vi_sc.show()
    print()

In [ ]:
# === Run all learning agents on each scenario ===

N_EP_SCENARIO = 500  # more episodes needed for larger, stochastic grids

agent_configs = {
    'Q-Learning':    dict(epsilon=0.15, gamma=0.95, alpha=0.1, seed=42),
    'SARSA':         dict(on_policy=True, epsilon=0.15, gamma=0.95, alpha=0.1, seed=42),
    'Dyna k=50':     dict(k_sim=50, epsilon=0.15, gamma=0.95, alpha=0.1, seed=42),
    'PriSweep k=50': dict(k_sim=50, prioritized=True, epsilon=0.15, gamma=0.95, alpha=0.1, seed=42),
}

scenario_results = {}  # (scenario_name, agent_name) -> (agent, steps, vstart)

for sc_name, sc_env in scenarios:
    print(f"\n{'='*60}")
    print(f"  {sc_name}")
    print(f"{'='*60}")
    for ag_name, kwargs in agent_configs.items():
        agent = TabularAgent(sc_env, **kwargs)
        v_start = []
        steps_list = []
        for _ in range(N_EP_SCENARIO):
            steps = agent.run_episode()
            steps_list.append(steps)
            v_start.append(np.max(agent.Q[sc_env.start]))
        scenario_results[(sc_name, ag_name)] = (agent, steps_list, v_start)
        print(f"  {ag_name:20s}  total={agent.total_steps:6d}  "
              f"last10_avg={np.mean(steps_list[-10:]):5.1f}")

print("\nDone.")

In [ ]:
# === Learning curves for all three scenarios (one row per scenario) ===

fig, axes = plt.subplots(3, 2, figsize=(16, 14))

colors = {
    'Q-Learning':    'tab:blue',
    'SARSA':         'tab:orange',
    'Dyna k=50':     'tab:green',
    'PriSweep k=50': 'tab:red',
}
window = 20

for row, (sc_name, sc_env) in enumerate(scenarios):
    vi_sc = vi_results[sc_name]
    vi_opt = vi_sc.evaluate_policy(n_episodes=1, max_steps=300)[0]
    vi_vs = vi_sc.V[sc_env.start]

    # Left: steps per episode (smoothed)
    ax_steps = axes[row, 0]
    for ag_name in agent_configs:
        _, steps, _ = scenario_results[(sc_name, ag_name)]
        smoothed = np.convolve(steps, np.ones(window)/window, mode='valid')
        ax_steps.plot(range(window-1, N_EP_SCENARIO), smoothed,
                      color=colors[ag_name], linewidth=2, label=ag_name)
    ax_steps.axhline(y=vi_opt, color='gray', linestyle=':', linewidth=2,
                     label=f'VI optimal ({vi_opt})')
    ax_steps.set_ylabel('Steps (smoothed)')
    ax_steps.set_title(f'{sc_name} — Steps per Episode')
    ax_steps.legend(fontsize=7)
    ax_steps.set_ylim(bottom=0)

    # Right: V*(start)
    ax_v = axes[row, 1]
    for ag_name in agent_configs:
        _, _, vstart = scenario_results[(sc_name, ag_name)]
        ax_v.plot(vstart, color=colors[ag_name], linewidth=2, label=ag_name)
    ax_v.axhline(y=vi_vs, color='gray', linestyle=':', linewidth=2,
                 label=f'VI optimal ({vi_vs:.2f})')
    ax_v.set_ylabel('V*(start)')
    ax_v.set_title(f'{sc_name} — Value Propagation')
    ax_v.legend(fontsize=7)

axes[2, 0].set_xlabel('Episode')
axes[2, 1].set_xlabel('Episode')
plt.tight_layout()
plt.show()

In [ ]:
# === Final policy grids for each scenario ===

for sc_name, sc_env in scenarios:
    vi_sc = vi_results[sc_name]
    print("=" * 80)
    print(f"  {sc_name}")
    print("=" * 80)
    print("\nVI (ground truth):")
    vi_sc.show()
    for ag_name in agent_configs:
        agent, _, _ = scenario_results[(sc_name, ag_name)]
        print(f"\n{ag_name}:")
        agent.show()
    print()

### Scenario takeaways

**A — The Gauntlet:**
- VI computes the risk-weighted optimal path and may route through or around the
  traps depending on expected cost.  With 40% trap probability and slip, the safe
  route often wins.
- Q-learning (off-policy) tends to learn a more aggressive path than SARSA, since
  it assumes perfect future play.  SARSA accounts for its own exploration slips
  and gives traps a wider berth.

**B — The Minefield:**
- With 20% action slip AND scattered stochastic traps, this is the hardest environment
  for all agents.  High variance in step counts reflects the randomness.
- Dyna and PriSweep converge faster because they replay from a model that captures
  the *average* outcome — smoothing out the noise that slows pure Q-learning.
- Compare V*(start) across agents: Q-learning may overshoot VI's value (overestimation
  from noisy max), while SARSA undershoots (accounts for exploration).

**C — The Maze:**
- No stochasticity — purely about how fast value propagates through long corridors.
- PriSweep should converge fastest: each update near the goal cascades backward
  through the corridor in a single sweep, while random Dyna wastes updates on
  already-converged cells.
- Q-learning without replay is slowest — value must propagate one step per episode.

---

## Try it yourself

Experiment with different settings to build intuition:

- **Change the grid**: add more traps, move the goal, make it 8x8.
- **Tune hyperparameters**: try higher/lower ε, α, γ.
- **Cliff walking**: put a row of traps along one side and compare
  Q-learning (walks the cliff edge) vs. SARSA (takes the safe route).
- **Larger k_sim**: does Dyna k=200 beat k=50?  At what point do
  diminishing returns kick in?

In [ ]:
# --- Playground cell: modify and re-run! ---

my_env = GridWorld(
    rows=8, cols=8,
    goal=(0, 7),
    traps={(1, 5), (1, 6), (1, 7), (2, 5)},
    start=(7, 0),
    goal_reward=10.0,
    trap_reward=-10.0,
    step_cost=-0.1,
)
my_env.show_grid()

# Solve with VI first (ground truth)
my_vi = ValueIteration(my_env, gamma=0.95)
my_vi.solve()
print("Optimal policy:")
my_vi.show()

# Then try a learning agent — same class, just different flags!
my_ql = TabularAgent(my_env, epsilon=0.1, gamma=0.95, alpha=0.1, seed=0)
steps = my_ql.run_episodes(300)
print(f"Q-Learning learned in {my_ql.total_steps} total steps:")
my_ql.show()